# 第8回：分類モデルの練習 — ペンギンデータで学ぶ評価と可視化

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session08_penguin_classification.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

第7回では、タイタニックデータを使って、決定木・Logistic回帰・複数モデルの比較を学びました。今回は、**ペンギンデータ**を使って、同じ流れを自分で実装する練習をします。

今回は説明を読みながらコードをそのまま実行するのではなく、**ヒントをもとに自分でコードを完成させる**形式です。

---

## 📋 本日の目標

1. ペンギンデータを分類問題として前処理できる
2. Logistic回帰・SVM・Random Forest・Neural Network を実装できる
3. 混同行列・classification report を使ってモデルを評価できる
4. ROC曲線とAUCを使って分類性能を比較できる
5. 分類境界を可視化し、モデルごとの違いを説明できる

---

## 0. 準備：ライブラリの読み込み

まず、必要なライブラリを読み込みます。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import subprocess

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import DecisionBoundaryDisplay

sns.set_theme(style="whitegrid")

# Google Colab で matplotlib の日本語が文字化けしないようにする
try:
    import japanize_matplotlib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "japanize-matplotlib", "-q"])
    import japanize_matplotlib

plt.rcParams["font.family"] = "IPAexGothic"
plt.rcParams["axes.unicode_minus"] = False

print("ライブラリの読み込み完了！")


---

## 1. ペンギンデータの読み込み

seaborn に入っている `penguins` データを使います。

### 1-1. データを読み込む


In [ ]:
df = sns.load_dataset("penguins")

# ヒント：データサイズと先頭5行を確認する
# print(df.shape)
# df.head()


### 1-2. 欠損値とクラス数を確認する


In [ ]:
# ヒント：欠損値の数を確認する
# df.isnull().sum()


In [ ]:
# ヒント：species, island, sex の分布を確認する
# df["species"].value_counts()


---

## 2. 2値分類データを作る

ペンギンデータには3種類のペンギンが含まれています。

- Adelie
- Chinstrap
- Gentoo

ROC曲線やAUCは、まず2値分類で練習するのが分かりやすいです。今回は `Adelie` と `Chinstrap` の2種類だけを使い、次の分類問題を作ります。

```
Adelie    → 1
Chinstrap → 0
```

つまり、「このペンギンは Adelie かどうか」を予測します。

### 2-1. Adelie と Chinstrap だけに絞る


In [ ]:
# ヒント：species が Adelie または Chinstrap の行だけを取り出す
df_bin = df[(df["species"] == "Adelie") | (df["species"] == "Chinstrap")].copy()

# ヒント：欠損値を削除する
df_bin = df_bin.dropna()

# ヒント：まず全員を Chinstrap(0) とし、Adelie の行だけ 1 にする
df_bin["target"] = 0
df_bin.loc[df_bin["species"] == "Adelie", "target"] = 1

# 確認
df_bin[["species", "target"]].head()


### 2-2. 説明変数 X と目的変数 y を作る

今回は数値特徴量だけを使います。


In [ ]:
feature_cols = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]

# ヒント：X には feature_cols の列、y には target を入れる
X = ...
y = ...

print("X:", X.shape)
print("y:", y.shape)


### 2-3. 訓練データとテストデータに分割する


In [ ]:
# ヒント：第7回と同じ train_test_split を使う
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"訓練データ: {X_train.shape[0]} 件")
print(f"テストデータ: {X_test.shape[0]} 件")


---

## 3. データを可視化する

モデルを作る前に、特徴量とクラスの関係を見ます。

### 3-1. 散布図


In [ ]:
# ヒント
# - data は df_bin
# - x は bill_length_mm
# - y は bill_depth_mm
# - hue と style は species

plt.figure(figsize=(..., ...))
sns.scatterplot(
    data=...,
    x=...,
    y=...,
    hue=...,
    style=...,
    s=...,
)
plt.xlabel(...)
plt.ylabel(...)
plt.title(...)
plt.tight_layout()
plt.show()


### 3-2. ペアプロット


In [ ]:
# ヒント
# - vars は feature_cols
# - hue は species

sns.pairplot(
    ...,
    vars=...,
    hue=...,
    diag_kind=...,
)
plt.show()


---

## 4. Logistic回帰

Logistic回帰は、特徴量の重み付き合計から線形スコアを作り、それを0〜1の確率に変換する分類モデルです。分類境界は基本的に直線的です。

### 4-1. モデルを学習する


In [ ]:
# ヒント：LogisticRegression を作成し、fit で学習する
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(..., ...)


### 4-2. 予測と評価


In [ ]:
# ヒント：predict で 0/1 の予測、predict_proba で確率を取得する
y_pred_lr = lr.predict(...)
y_score_lr = lr.predict_proba(...)[:, 1]

print("正解率:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=["Chinstrap(0)", "Adelie(1)"]))


### 4-3. 混同行列


In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

# ヒント：display_labels には ["Chinstrap(0)", "Adelie(1)"] を指定する
ConfusionMatrixDisplay(..., display_labels=...).plot(
    cmap=...,
    colorbar=...,
)
plt.title(...)
plt.tight_layout()
plt.show()


---

## 5. SVM

SVM（Support Vector Machine）は、2つのクラスを分ける境界を探すモデルです。特に、境界とデータ点の間の余裕（マージン）を大きくするように学習します。

`kernel="rbf"` を使うと、直線だけでなく曲線的な境界も作れます。ROC曲線を描くため、今回は `probability=True` を指定します。


In [ ]:
# ヒント：SVC を使う。ROC用に probability=True を指定する
svm = SVC(kernel="rbf", probability=True, random_state=42)
svm.fit(..., ...)

y_pred_svm = svm.predict(...)
y_score_svm = svm.predict_proba(...)[:, 1]

print("正解率:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm, target_names=["Chinstrap(0)", "Adelie(1)"]))


---

## 6. Random Forest

Random Forest は、決定木をたくさん作り、その多数決で分類するモデルです。1本の決定木より安定しやすく、実務でもよく使われます。


In [ ]:
# ヒント：RandomForestClassifier を使う
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(..., ...)

y_pred_rf = rf.predict(...)
y_score_rf = rf.predict_proba(...)[:, 1]

print("正解率:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=["Chinstrap(0)", "Adelie(1)"]))


### 6-1. 特徴量重要度

Random Forest では、どの特徴量が分類に効いていたかを簡単に確認できます。


In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

# ヒント：importance_df を表示する
importance_df


In [ ]:
# ヒント
# - data は importance_df
# - x は importance
# - y は feature

plt.figure(figsize=(..., ...))
sns.barplot(data=..., x=..., y=...)
plt.xlabel(...)
plt.ylabel(...)
plt.title(...)
plt.tight_layout()
plt.show()


---

## 7. Neural Network

Neural Network は、複数の重み付き計算を層として重ねることで、複雑な関係を表現するモデルです。ここでは scikit-learn の `MLPClassifier` を使います。

注意点として、Neural Network は特徴量のスケールに影響されやすいです。本格的には標準化を使いますが、今回はまずそのまま実行して、他のモデルと比較してみます。


In [ ]:
# ヒント：MLPClassifier を使う
nn = MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=1000, random_state=42)
nn.fit(..., ...)

y_pred_nn = nn.predict(...)
y_score_nn = nn.predict_proba(...)[:, 1]

print("正解率:", accuracy_score(y_test, y_pred_nn))
print(classification_report(y_test, y_pred_nn, target_names=["Chinstrap(0)", "Adelie(1)"]))


---

## 8. 複数モデルをまとめて比較する

ここまでのモデルは、どれも同じ流れで使えます。

```
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

### 8-1. for 文で正解率を比較する


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=1000, random_state=42),
}

results = []

for name, model in models.items():
    # ヒント：fit → predict → accuracy_score の順に書く
    model.fit(..., ...)
    y_pred = model.predict(...)
    acc = accuracy_score(..., ...)

    results.append({
        "model": name,
        "accuracy": acc,
    })

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df


### 8-2. 正解率を棒グラフにする


In [ ]:
# ヒント
# - data は results_df
# - x は accuracy
# - y は model

plt.figure(figsize=(..., ...))
sns.barplot(data=..., x=..., y=...)
plt.xlabel(...)
plt.ylabel(...)
plt.title(...)
plt.xlim(..., ...)
plt.tight_layout()
plt.show()


---

## 9. ROC曲線とAUC

正解率は、分類の閾値を1つに決めた後の評価です。一方、ROC曲線は、閾値を変えたときにモデルの性能がどう変わるかを見る方法です。

- **TPR**：実際に Adelie の個体を、Adelie と見つけられた割合
- **FPR**：実際には Chinstrap なのに、Adelie と誤って予測した割合
- **AUC**：ROC曲線の下の面積。1に近いほどよい。0.5に近いとランダムに近い

### 9-1. Logistic回帰のROC曲線


In [ ]:
# ヒント：y_score_lr は Logistic回帰が出した「Adelieである確率」
fpr, tpr, thresholds = roc_curve(..., ...)
auc_lr = roc_auc_score(..., ...)

plt.figure(figsize=(..., ...))
plt.plot(..., ..., linewidth=2, label=f"Logistic回帰 AUC = {auc_lr:.3f}")
plt.plot(..., ..., linestyle=..., color=..., label=...)
plt.xlabel(...)
plt.ylabel(...)
plt.title(...)
plt.legend()
plt.grid(..., alpha=...)
plt.tight_layout()
plt.show()


### 9-2. 複数モデルのROC曲線


In [ ]:
plt.figure(figsize=(..., ...))

for name, model in models.items():
    # ヒント：学習する
    model.fit(..., ...)

    # ヒント：predict_proba で Adelie(1) の確率を取り出す
    y_score = model.predict_proba(...)[:, 1]

    # ヒント：AUC と ROC曲線用の fpr, tpr を求める
    auc = roc_auc_score(..., ...)
    fpr, tpr, _ = roc_curve(..., ...)

    plt.plot(..., ..., linewidth=2, label=f"{name} AUC={auc:.3f}")

plt.plot(..., ..., linestyle=..., color=..., label=...)
plt.xlabel(...)
plt.ylabel(...)
plt.title(...)
plt.legend(fontsize=...)
plt.grid(..., alpha=...)
plt.tight_layout()
plt.show()


### 9-3. 閾値を変えて混同行列を見る

Logistic回帰の予測確率を使い、閾値を変えて予測結果がどう変わるか確認します。


In [ ]:
threshold_list = [0.3, 0.5, 0.7]

for threshold in threshold_list:
    # ヒント：確率が threshold 以上なら 1、それ以外なら 0
    y_pred_custom = (... >= threshold).astype(int)

    print(f"\n=== 閾値 {threshold} ===")
    print("正解率:", accuracy_score(..., ...))
    print(confusion_matrix(..., ...))


---

## 10. Visualization：分類境界を見る

最後に、`bill_length_mm` と `bill_depth_mm` の2特徴量だけを使って、モデルごとの分類境界を可視化します。


In [ ]:
X_train_2d = X_train[["bill_length_mm", "bill_depth_mm"]]
y_train_2d = y_train

boundary_models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=1000, random_state=42),
}

fig, axes = plt.subplots(..., ..., figsize=(..., ...))
axes = axes.ravel()

for ax, (name, model) in zip(axes, boundary_models.items()):
    # ヒント：2次元の特徴量で学習する
    model.fit(..., ...)

    DecisionBoundaryDisplay.from_estimator(
        ...,
        ...,
        response_method=...,
        cmap=...,
        alpha=...,
        ax=...,
    )

    ax.scatter(
        ...,
        ...,
        c=...,
        cmap=...,
        edgecolors=...,
        s=...,
        alpha=...,
    )
    ax.set_title(...)
    ax.set_xlabel(...)
    ax.set_ylabel(...)

plt.tight_layout()
plt.show()


---

## 📝 演習

### 演習 1：前処理を完成させる

`X = ...` と `y = ...` を埋め、訓練データとテストデータを作成してください。`X_train.shape` と `X_test.shape` を確認しましょう。

### 演習 2：4つのモデルを実装する

Logistic回帰・SVM・Random Forest・Neural Network の4つについて、`fit`、`predict`、`classification_report` を実行してください。

### 演習 3：for 文で比較する

`models` 辞書を使って、4モデルの正解率を表にまとめてください。どのモデルの正解率が最も高かったですか？

### 演習 4：ROC曲線を比較する

4モデルのROC曲線を同じ図に描き、AUCが最も高いモデルを確認してください。正解率が最も高いモデルと同じでしたか？

### 演習 5：分類境界を説明する

分類境界の図を見て、以下を答えてください。

1. 境界が最も直線的なモデルはどれですか？
2. 境界が最も複雑なモデルはどれですか？
3. 複雑な境界は、必ず良いモデルと言えますか？

---

## 🏆 本日の総合課題

以下を実行し、結果と考察をコードのコメントとして記録してください。


In [ ]:
# 1. ペンギンデータを読み込み、Adelie と Chinstrap の2値分類データを作る
# 2. Logistic回帰・SVM・Random Forest・Neural Network を学習する
# 3. 4モデルの正解率・classification_report・混同行列を比較する
# 4. 4モデルのROC曲線とAUCを比較する
# 5. 分類境界を可視化し、モデルごとの違いを説明する
# 6. 最終的にどのモデルを選ぶかを、正解率・recall・AUC の観点から説明する


---

## まとめ

今回のポイントは以下です。

- タイタニックで学んだ分類の流れは、ペンギンデータにも同じように使える
- scikit-learn では、モデルを入れ替えても `fit` と `predict` の流れはほとんど同じ
- Logistic回帰・SVM・Random Forest・Neural Network は、分類境界の形や解釈しやすさが異なる
- 正解率だけでなく、混同行列・classification report・ROC曲線・AUC を使うと評価が深くなる
- 可視化すると、モデルごとの分類の仕方が理解しやすい
